<a href="https://colab.research.google.com/github/njshrutics24-bot/BIS_LAB/blob/main/AntColonyOptimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# N J SHRUTI
#1BF24CS185
import random
import math

WAREHOUSE = 0
CUSTOMERS = list(range(1, 7))

PACKAGE_WEIGHTS = {1: 3, 2: 4, 3: 2, 4: 5, 5: 3, 6: 2}

NUM_AGENTS = 2
AGENT_CAPACITY = 10

MAX_ITERATIONS = 50
COLONY_SIZE = 20
EVAPORATION_RATE = 0.2
ALPHA = 1.0
BETA = 2.0
CONSTANT = 100.0

COORDS = {
    0: (50, 50),
    1: (20, 80),
    2: (30, 75),
    3: (80, 80),
    4: (85, 20),
    5: (25, 20),
    6: (45, 35)
}

num_nodes = len(COORDS)
travel_time = {}
pheromone = {}

for u in range(num_nodes):
    travel_time[u] = {}
    pheromone[u] = {}
    for v in range(num_nodes):
        if u == v:
            travel_time[u][v] = 0.0
        else:
            x1, y1 = COORDS[u]
            x2, y2 = COORDS[v]
            travel_time[u][v] = round(math.hypot(x2 - x1, y2 - y1), 1)
        pheromone[u][v] = 1.0


def dynamic_aco_delivery():
    best_routes = None
    best_time = float('inf')

    for iteration in range(MAX_ITERATIONS):
        wave_solutions = []

        for ant in range(COLONY_SIZE):
            routes, total_time = build_ant_routes()
            wave_solutions.append((routes, total_time))

            if total_time < best_time:
                best_time = total_time
                best_routes = routes

        for u in range(num_nodes):
            for v in range(num_nodes):
                pheromone[u][v] = pheromone[u][v] * (1 - EVAPORATION_RATE)

        for routes, total_time in wave_solutions:
            reward = CONSTANT / total_time
            for agent_route in routes:
                for step in range(len(agent_route) - 1):
                    u = agent_route[step]
                    v = agent_route[step + 1]
                    pheromone[u][v] += reward

    return best_routes, round(best_time, 2)


def build_ant_routes():
    unvisited = set(CUSTOMERS)
    fleet_routes = []
    total_time = 0.0

    for agent in range(NUM_AGENTS):
        current_loc = WAREHOUSE
        current_load = 0
        route = [WAREHOUSE]

        while unvisited:
            valid_customers = []
            for c in unvisited:
                if current_load + PACKAGE_WEIGHTS[c] <= AGENT_CAPACITY:
                    valid_customers.append(c)

            if not valid_customers:
                break

            scores = []
            for next_node in valid_customers:
                tau = pheromone[current_loc][next_node] ** ALPHA
                eta = (1.0 / travel_time[current_loc][next_node]) ** BETA
                scores.append(tau * eta)

            total_score = sum(scores)
            probabilities = [s / total_score for s in scores]
            chosen_list = random.choices(valid_customers, weights=probabilities, k=1)
            next_stop = chosen_list.pop()

            route.append(next_stop)
            total_time += travel_time[current_loc][next_stop]
            current_load += PACKAGE_WEIGHTS[next_stop]
            current_loc = next_stop
            unvisited.remove(next_stop)

        route.append(WAREHOUSE)
        total_time += travel_time[current_loc][WAREHOUSE]
        fleet_routes.append(route)

    if unvisited:
        total_time += len(unvisited) * 9999

    return fleet_routes, total_time


def on_live_event(event_type, road_u=None, road_v=None, new_customer=None, new_weight=0):
    print(f"\n[ALERT] Live Event Triggered: {event_type}")

    if event_type == "TRAFFIC_JAM":
        travel_time[road_u][road_v] *= 5
        travel_time[road_v][road_u] *= 5
        pheromone[road_u][road_v] = 0.01
        pheromone[road_v][road_u] = 0.01
        print(f"Road ({road_u} <-> {road_v}) heavily congested! Pheromones wiped.")

    elif event_type == "NEW_ORDER":
        CUSTOMERS.append(new_customer)
        PACKAGE_WEIGHTS[new_customer] = new_weight
        print(f"New Order added for Customer {new_customer} (Weight: {new_weight}).")

    updated_routes, updated_time = dynamic_aco_delivery()
    return updated_routes, updated_time


if __name__ == "__main__":
    random.seed(42)

    routes, time = dynamic_aco_delivery()
    print("=== INITIAL DISPATCH PLAN ===")
    for i, r in enumerate(routes, 1):
        print(f"Agent {i} Route: {r}")
    print(f"Total Fleet Travel Time: {time} mins")

    new_routes, new_time = on_live_event("TRAFFIC_JAM", road_u=0, road_v=6)
    print("\n=== UPDATED PLAN AFTER TRAFFIC JAM ===")
    for i, r in enumerate(new_routes, 1):
        print(f"Agent {i} Route: {r}")
    print(f"New Total Fleet Travel Time: {new_time} mins")


=== INITIAL DISPATCH PLAN ===
Agent 1 Route: [0, 6, 5, 4, 0]
Agent 2 Route: [0, 2, 1, 3, 0]
Total Fleet Travel Time: 292.5 mins

[ALERT] Live Event Triggered: TRAFFIC_JAM
Road (0 <-> 6) heavily congested! Pheromones wiped.

=== UPDATED PLAN AFTER TRAFFIC JAM ===
Agent 1 Route: [0, 2, 1, 3, 0]
Agent 2 Route: [0, 4, 6, 5, 0]
New Total Fleet Travel Time: 298.5 mins
